## Bài 1: Thiết lập Pipeline chống rò rỉ dữ liệu và Mô hình cơ sở 

- **Kiểm tra dữ liệu đầu vào:** Nạp `Loan_approval_data_2025.csv`, loại bỏ cột định danh `customer_id`.
- **Phân chia dữ liệu 70/15/15:** Tách phân tầng thành **Train (70%)**, **Validation (15%)** và **Test (15%)** với `random_state = 42`.
- **Đóng gói `ColumnTransformer` & `Pipeline`:** Chỉ `fit` phép chuẩn hóa `StandardScaler` và `OneHotEncoder` trên tập huấn luyện, triệt tiêu hoàn toàn rò rỉ dữ liệu.
- **Thiết lập Mô hình cơ sở (`DummyClassifier`):** Xác lập mức sàn tối thiểu để đối chiếu các thuật toán ở Bài 2.

### 1. Nạp dữ liệu và kiểm tra phân bố nhãn mục tiêu (`loan_status`)

In [1]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

# Đồng bộ giao diện và bảng màu với Kiemtra1
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
PALETTE_4 = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12"]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = Path("../data/Loan_approval_data_2025.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/Loan_approval_data_2025.csv")
assert DATA_PATH.exists(), f"Không tìm thấy dữ liệu tại {DATA_PATH.resolve()}"

IMG_DIR = Path("../images")
MODEL_DIR = Path("../models")
IMG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def get_model_score(estimator, X_eval):
    '''Trích xuất điểm liên tục (xác suất lớp 1 hoặc khoảng cách decision_function).'''
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_eval)[:, 1]
    return estimator.decision_function(X_eval)


def evaluate(y_true, y_pred, y_score):
    '''Tính bộ chỉ số đánh giá đa chiều cho bài toán phân loại tín dụng.'''
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "ROC-AUC": roc_auc_score(y_true, y_score),
        "F1-Macro": f1_score(y_true, y_pred, average="macro"),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "PR-AUC": average_precision_score(y_true, y_score),
    }


def load_and_split_data():
    '''Nạp dữ liệu, loại bỏ customer_id và chia phân tầng 70% Train / 15% Val / 15% Test.'''
    df = pd.read_csv(DATA_PATH)
    X = df.drop(columns=["customer_id", "loan_status"])
    y = df["loan_status"]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
    )

    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            (
                "cat",
                OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
                cat_cols,
            ),
        ],
        remainder="drop",
    )
    return df, X, y, X_train, X_val, X_test, y_train, y_val, y_test, num_cols, cat_cols, preprocessor


def build_pipelines(preprocessor):
    '''Đóng gói mô hình cơ sở và các thuật toán Bài 4 & Bài 5 vào Pipeline thống nhất.'''
    models = {
        "Dummy (Baseline đa số)": DummyClassifier(strategy="most_frequent"),
        "Perceptron (PLA)": Perceptron(max_iter=1000, tol=1e-3, random_state=RANDOM_SEED),
        "SGDClassifier (Log-loss + GD)": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-4,
            max_iter=1000,
            tol=1e-4,
            random_state=RANDOM_SEED,
        ),
        "Logistic Regression (Mặc định)": LogisticRegression(
            max_iter=1000, C=1.0, random_state=RANDOM_SEED
        ),
        "Logistic Regression (Balanced)": LogisticRegression(
            max_iter=1000, C=1.0, class_weight="balanced", random_state=RANDOM_SEED
        ),
    }
    return {
        name: Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
        for name, clf in models.items()
    }

df, X, y, X_train, X_val, X_test, y_train, y_val, y_test, num_cols, cat_cols, preprocessor = load_and_split_data()

n_approved = int((y == 1).sum())
n_rejected = int((y == 0).sum())

print("=" * 80)
print("THÔNG TIN TẬP DỮ LIỆU GỐC (BÀI KIỂM TRA 2)")
print("=" * 80)
print(f" - Tập dữ liệu        : {DATA_PATH}")
print(f" - Kích thước bảng    : {df.shape[0]:,} dòng x {df.shape[1]} cột")
print(f" - Đặc trưng đầu vào   : {X.shape[1]} cột ({len(num_cols)} số + {len(cat_cols)} phân loại)")
print(f" - Lớp 1 (Được duyệt) : {n_approved:,} ({n_approved / len(y):.2%})")
print(f" - Lớp 0 (Bị từ chối) : {n_rejected:,} ({n_rejected / len(y):.2%})")
display(X.head(3))

THÔNG TIN TẬP DỮ LIỆU GỐC (BÀI KIỂM TRA 2)
 - Tập dữ liệu        : ..\data\Loan_approval_data_2025.csv
 - Kích thước bảng    : 50,000 dòng x 20 cột
 - Đặc trưng đầu vào   : 18 cột (15 số + 3 phân loại)
 - Lớp 1 (Được duyệt) : 27,523 (55.05%)
 - Lớp 0 (Bị từ chối) : 22,477 (44.95%)


,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio
0,40,Employed,17.2,25579,692,5.3,895,10820,0,0,0,Credit Card,Business,600,17.02,0.423,0.023,0.008
1,33,Employed,7.3,43087,627,3.5,169,16550,0,1,0,Personal Loan,Home Improvement,53300,14.10,0.384,1.237,0.412
2,42,Student,1.1,20840,689,8.4,17,7852,0,0,0,Credit Card,Debt Consolidation,2100,18.33,0.377,0.101,0.034


- **Thống kê:** Toàn bộ tập dữ liệu gồm $50.000$ hồ sơ vay với $18$ đặc trưng đầu vào hợp lệ (`customer_id` đã bị loại bỏ ngay từ đầu).
- **Ý nghĩa mô hình hóa:** Tỷ lệ nhãn $55{,}05\%$ được duyệt so với $44{,}95\%$ bị từ chối bảo đảm tính cân đối tương đối, song vẫn cần theo dõi song song `ROC-AUC`, `F1-Macro`, `Precision` và `Recall` theo đúng bộ tiêu chí đã xác lập.

### 2. Phân chia tập 70/15/15 và Đóng gói `ColumnTransformer`

In [2]:
# Kiểm tra kích thước 3 tập và số chiều sau ColumnTransformer (chỉ fit thử trên X_train)
X_train_tr = preprocessor.fit_transform(X_train)
ohe_names = preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_names

split_summary = pd.DataFrame(
    [
        {"Tập dữ liệu": "Huấn luyện (Train)", "Số mẫu": len(X_train), "Tỷ lệ (%)": len(X_train) / len(X) * 100, "Tỷ lệ duyệt (y=1)": y_train.mean()},
        {"Tập dữ liệu": "Kiểm định (Val)", "Số mẫu": len(X_val), "Tỷ lệ (%)": len(X_val) / len(X) * 100, "Tỷ lệ duyệt (y=1)": y_val.mean()},
        {"Tập dữ liệu": "Kiểm tra (Test)", "Số mẫu": len(X_test), "Tỷ lệ (%)": len(X_test) / len(X) * 100, "Tỷ lệ duyệt (y=1)": y_test.mean()},
    ]
)

print("=" * 80)
print("PHÂN CHIA DỮ LIỆU 70/15/15 VÀ TIỀN XỬ LÝ KHÉP KÍN")
print("=" * 80)
display(split_summary.round(4))
print(f"Số chiều đặc trưng sau OneHotEncoder (drop='first'): {X_train_tr.shape[1]} chiều")

PHÂN CHIA DỮ LIỆU 70/15/15 VÀ TIỀN XỬ LÝ KHÉP KÍN


,Tập dữ liệu,Số mẫu,Tỷ lệ (%),Tỷ lệ duyệt (y=1)
0,Huấn luyện (Train),35000,70.0,0.5505
1,Kiểm định (Val),7500,15.0,0.5505
2,Kiểm tra (Test),7500,15.0,0.5504


Số chiều đặc trưng sau OneHotEncoder (drop='first'): 24 chiều


- **Chống rò rỉ dữ liệu:** Tập Test ($7.500$ mẫu -- $15\%$) được cô lập hoàn toàn từ đầu; các tham số trung bình, độ lệch chuẩn của `StandardScaler` và danh mục của `OneHotEncoder` chỉ được học trên tập Train bên trong `Pipeline`.
- **Bảo toàn tỷ lệ nhãn:** Cả ba tập Train ($35.000$), Val ($7.500$) và Test ($7.500$) giữ nguyên tỷ lệ nhãn dương xấp xỉ $55{,}05\%$ nhờ phân tầng `stratify=y`.

### 3. Xây dựng và đánh giá Mô hình cơ sở (`DummyClassifier` -- Majority Class)

In [3]:
baseline_pipe = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_res = cross_validate(
    baseline_pipe,
    X_train,
    y_train,
    cv=cv,
    scoring=["roc_auc", "f1_macro", "accuracy", "precision", "recall"],
    n_jobs=-1,
)

baseline_pipe.fit(X_train, y_train)
y_val_pred = baseline_pipe.predict(X_val)
y_val_score = get_model_score(baseline_pipe, X_val)
val_metrics = evaluate(y_val, y_val_pred, y_val_score)

print("=" * 80)
print("MỨC SÀN MÔ HÌNH CƠ SỞ (DUMMY CLASSIFIER - LỚP ĐA SỐ)")
print("=" * 80)
print(f" - 5-Fold CV ROC-AUC  : {cv_res['test_roc_auc'].mean():.4f} ± {cv_res['test_roc_auc'].std():.4f}")
print(f" - 5-Fold CV F1-Macro : {cv_res['test_f1_macro'].mean():.4f} ± {cv_res['test_f1_macro'].std():.4f}")
print(f" - 5-Fold CV Accuracy : {cv_res['test_accuracy'].mean():.4f} ± {cv_res['test_accuracy'].std():.4f}")
print(f" - Validation ROC-AUC : {val_metrics['ROC-AUC']:.4f}")
print(f" - Validation Spec.   : {val_metrics['Specificity']:.4f} (Recall lớp 0 = 0%)")

MỨC SÀN MÔ HÌNH CƠ SỞ (DUMMY CLASSIFIER - LỚP ĐA SỐ)
 - 5-Fold CV ROC-AUC  : 0.5000 ± 0.0000
 - 5-Fold CV F1-Macro : 0.3550 ± 0.0000
 - 5-Fold CV Accuracy : 0.5505 ± 0.0001
 - Validation ROC-AUC : 0.5000
 - Validation Spec.   : 0.0000 (Recall lớp 0 = 0%)


- **Mức sàn tham chiếu:** Do luôn gán nhãn lớp đa số ($\hat{y} = 1$), `DummyClassifier` đạt Accuracy $55{,}05\%$ và Recall $1{,}0000$ một cách tầm thường, nhưng `ROC-AUC = 0,5000`, `F1-Macro = 0,3550` và `Specificity = 0,0000` (không lọc được bất kỳ hồ sơ nợ xấu nào).
- **Ý nghĩa đối chứng:** Đây là mốc tối thiểu chứng minh mọi mô hình học máy ở Bài 2 phải vượt trội cả về khả năng phân tách xác suất (`ROC-AUC`) lẫn cân bằng hai lớp (`F1-Macro`).